# Graph building blocks: examples

Runnable versions of the `PLAN.md` §3.3 examples that the current code supports. Everything runs on CPU with random data, so the numbers show the code works, not that a model is any good.

1. Wrap any model as a one-node graph
2. Train it with your own loop
3. Swap one part; nothing else changes
4. Deploy and predict on one image
5. Mistakes fail early, such as a bad name
6. Record a run
7. Evaluate top-1 and top-5

Examples that need code which doesn't exist yet (`Trainer`, data modules, frozen teachers) are still in `PLAN.md` §3.3–3.4. Move each one here when its code lands.

This workbook is tested in Python 3.14. 

Run from `examples/`. `lwfm` isn't an installed package yet, so put the repository root on the import path.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import tempfile

import torch
from torch import nn
from torchvision.models import resnet18
from torchvision.transforms import v2

from lwfm.core.graph import ExecutionGraph, GraphError, ModelNode
from lwfm.core.losses import LossBinding, LossComposer

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)  # ImageNet statistics
_ = torch.manual_seed(0)

## 1. Wrap any model as a one-node graph

The model is an untouched torchvision `resnet18`. `inputs` lists the values passed to `forward`, in order; `"inputs.image"` means `batch["inputs"]["image"]`. `outputs` names what it returns; the node also stores the output under its own name, as `student.logits`. 

The model is a straight chain, so we can simply use plain torch functions. We can use `nn.Sequential` to bundle together some preprocessing as well as ResNet18. Each of these is an `nn.Module`. The torchvision preprocessing, `v2.Resize` and `v2.Normalize`, resize the 96x96 images to 64x64 and then normalise (respectively). `lwfm` adds none of its own preprocessing, so this step can contain any preprocessing as required. 

In most use cases, the pure torch function `nn.Sequential` is sufficient. We only need to use `lwfm`'s graph where values need to branch or merge.

Passing `preflight_sample=batch` runs the graph once when it is built, to check shapes (section 5).

In [3]:
batch = {
    "inputs": {"image": torch.rand(8, 3, 96, 96)},  # floats in [0, 1]
    "targets": {"class": torch.randint(0, 10, (8,))},
    "meta": {"sample_id": [f"s{i}" for i in range(8)]},
}

# We can set up a simple ResNet18 model, we'll call it student.
student = ModelNode(
    name="student",
    module=nn.Sequential(  # a straight chain: plain torch
        v2.Resize((64, 64), antialias=True),
        v2.Normalize(MEAN, STD),
        resnet18(num_classes=10),  # untouched torchvision model
    ),
    inputs="inputs.image",  # passed to forward as its first argument
    outputs="logits",  # its Tensor return, stored as "student.logits"
)
# preflight_sample: run once on this batch now, to check shapes
pipeline = ExecutionGraph([student], preflight_sample=batch)

# Check the output shape
out = pipeline(batch)
{name: tuple(value.shape) for name, value in out.items()}

{'student.logits': (8, 10)}

## 2. Train it with your own loop

As per the specs of the framework, the pipeline (an `ExecutionGraph`) and the loss composer are enough to train our moedl.

A `Trainer` will be a convenience, not a requirement. 

The loss reads `student.logits` and `targets.class`, passed to `CrossEntropyLoss(input, target)` in that order. We can perform a quick check to test this works by overfitting. Continuously training on the same batch should result in the loss falling towards 0 (gradients reach the weights).  

In [4]:
# Define what inputs/outputs our loss function reads from
CE_READS = ["student.logits", "targets.class"]

loss_fn = LossComposer([LossBinding("ce", nn.CrossEntropyLoss(), CE_READS)])

# Perform standard SGD
optimizer = torch.optim.SGD(pipeline.parameters(), lr=0.1)

# Fit the model
pipeline.train()

for step in range(20):  # overfit one batch: loss should fall
    total, terms = loss_fn(batch, pipeline(batch))
    optimizer.zero_grad()
    total.backward()
    optimizer.step()
    if step in (0, 19):
        print(step, {name: round(t.item(), 4) for name, t in terms.items()})

0 {'ce': 2.6651}
19 {'ce': 0.0006}


## 3. Swap one part; nothing else changes

**A different model.** `TinyViT` names its argument `pixels` and returns a dict. It is the last step of the chain, so its dict is what the node receives. Inputs are passed by position, so `inputs` stays the same; `outputs` lists the dict fields to keep. The loss still reads `student.logits`, so it needs no changes. The extra `cls_token` output becomes `student.cls_token`, ready for a later feature-distillation loss.

In [ ]:
class TinyViT(nn.Module):
    def __init__(self, num_classes: int, dim: int = 32, patch: int = 8):
        # Initialise the torch nn.Module
        super().__init__()
        # Convert image patches into tokens by performing a 2D Convolutional step
        self.embed = nn.Conv2d(
            # Our image has 3 colour channels (RGB)
            3,
            # Each patch is converted into 32 numbers by default (or dim)
            dim,
            # The convolutional kernel is 8x8 by default (or patch)
            kernel_size=patch,
            # This is the distance between patches
            stride=patch,
        )

        # Define a CLS token
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))

        # Create one transformer encoder layer,
        #   with 4 attention heads and 2* dim (default 64) hidden features
        layer = nn.TransformerEncoderLayer(
            dim, nhead=4, dim_feedforward=2 * dim, batch_first=True
        )
        # Stack two encoder layers
        self.encoder = nn.TransformerEncoder(
            layer, num_layers=2, enable_nested_tensor=False
        )

        # Create the classification head (convert CLS token into scores)
        self.head = nn.Linear(dim, num_classes)

    # Foreard pass an image through the model
    def forward(self, pixels: torch.Tensor) -> dict[str, torch.Tensor]:
        # Pass the images through the convolutional layer,
        #   flatten onto one dimension then transpose.
        tokens = self.embed(pixels).flatten(2).transpose(1, 2)  # [B, N, dim]

        # Add the CLS token to the beginning of the sequence
        tokens = torch.cat([self.cls.expand(len(tokens), -1, -1), tokens], dim=1)

        # Pass all 65 tokens to the Transformer encoder,
        #   and select the first token from each image
        cls_token = self.encoder(tokens)[:, 0]

        # Return raw class scores and CLS representation
        return {"logits": self.head(cls_token), "cls_token": cls_token}


vit_student = ModelNode(
    name="student",
    module=nn.Sequential(
        v2.Resize((64, 64), antialias=True),
        v2.Normalize(MEAN, STD),
        TinyViT(num_classes=10),  # Classify 10 different types of image
    ),
    inputs="inputs.image",
    outputs=["logits", "cls_token"],  # dict fields to keep
)

# Create a graph with just the ViT student and run a test batch
vit_pipeline = ExecutionGraph([vit_student], preflight_sample=batch)
{name: tuple(value.shape) for name, value in vit_pipeline(batch).items()}

{'student.logits': (8, 10), 'student.cls_token': (8, 32)}

**A different loss, or a weighted mix.** 

Let's define a new loss function, which is a weighted mix between Cross Entropy loss and Margin Loss:

$\mathcal{L} = 0.9\,\mathrm{CE}_{\text{smoothed}} + 0.1\,\mathrm{margin}$. 

Both are plain `torch.nn` losses, reading the same values. The composer returns the total for `backward()` and each term, detached, for logging.

In [6]:
mixed_loss = LossComposer(
    [
        LossBinding(
            "ce", nn.CrossEntropyLoss(label_smoothing=0.1), CE_READS, weight=0.9
        ),
        LossBinding("margin", nn.MultiMarginLoss(), CE_READS, weight=0.1),
    ]
)
total, terms = mixed_loss(batch, vit_pipeline(batch))
total, terms

(tensor(2.0463, grad_fn=<AddBackward0>),
 {'ce': tensor(2.2000), 'margin': tensor(0.6632)})

## 4. Deploy and predict on one image

`deployable_model` returns the node's module, here the `nn.Sequential` of preprocessing and model, as a plain `nn.Module`. It contains no graph, names or losses, and it carries its preprocessing, so it takes a raw `[0, 1]` image of any size. Its state dict holds only model weights (`Resize` and `Normalize` store none), so a fresh copy reloads it and gives identical predictions. A random tensor stands in for an image file. For a real image: `read_image(path, mode=ImageReadMode.RGB).float() / 255` from `torchvision.io`.

In [ ]:
# Get the deployable model for student
model = pipeline.deployable_model("student").eval()  # Resize, Normalize, resnet18

# Create a temporary directory to store model weights
with tempfile.TemporaryDirectory() as tmp:
    # Store the model weights in this file
    path = Path(tmp) / "model_weights.pt"
    # Save
    torch.save(model.state_dict(), path)

    # Create a fresh copy of the same model
    fresh = nn.Sequential(
        v2.Resize((64, 64), antialias=True),
        v2.Normalize(MEAN, STD),
        resnet18(num_classes=10),
    )

    # Load the saved weights, and evaluate
    fresh.load_state_dict(torch.load(path, weights_only=True))
    fresh.eval()

# Create a random image (RGB Channels x Height x Width)
image = torch.rand(3, 120, 80)  # [3, H, W] in [0, 1], any size

# Enter inference mode, e.g. only predictions, not training
with torch.inference_mode():
    # Make predictions and return probabilities and classes.
    # Convert logits into probabilities by using softmax on the last dimension
    #  (in this case 10 classes)
    # Find the five highest probabilities and return them
    probs, classes = model(image[None]).softmax(-1).topk(5)

    # Compare the fresh model with the existing model
    # This will output nothing and continue if the models are sufficiently close.
    # Otherwise, it will halt execution with an AssertionError.
    torch.testing.assert_close(fresh(image[None]), model(image[None]))

# Display the results
classes, probs

(tensor([[1, 7, 4, 0, 6]]), tensor([[0.3534, 0.2576, 0.2468, 0.1292, 0.0056]]))

## 5. Mistakes fail early, with the bad name

The graph checks its wiring when it is built, before any data is touched: every value a node reads must come from the batch (`inputs.`, `targets.` or `meta.`) or from another node, node names must exist, be unique, and there must be no cycles.

Batch data can be accessesd using `<group>.<key>` notation, for example `inputs.image`.
Node data can be accessed using `<node>.<output>`, for example `student.logits`.

Let's create an example where we read from a non-existent name (by typo) and we should expect a GraphError.

In [ ]:
try:
    ExecutionGraph(
        [
            ModelNode("student", resnet18(), inputs="inputs.image", outputs="logits"),
            ModelNode(
                # "studnet" is a deliberate typo: building fails and names it
                "head",
                nn.Linear(512, 10),
                inputs="studnet.features",
                outputs="logits",
            ),
        ]
    )
except GraphError as error:
    print(f"GraphError: {error}")
except Exception as error:
    print(f"Unexpected error {type(error).__name__}: {error}")

GraphError: 'head' reads 'studnet.features', which no node writes


Wrong **shapes** only show up when data flows. To mitigate errors when using real error, we run a small "preflight" which allows us to check data dimensions etc before supplying any real data.

To achieve this, set `preflight_sample` as follows: `ExecutionGraph(nodes, preflight_sample=batch)`. When lwfm builds the graph it also runs it once on that batch (the *preflight* check), in eval mode without gradients. Any batch works, even zeros of the right shape. It also checks every output keeps the batch dimension first. The output shapes are kept in `pipeline.preflight_output_shapes`, and `pipeline.preflight(batch)` re-runs the check. Without a sample the check is skipped. 

On failure the error produces a PyTorch error type, with a note naming the node and the shapes it read. The same note is added in ordinary `pipeline(batch)` calls, and losses and metrics add one naming themselves.

In [9]:
# Import python tracebacks, to allow us to inspect errors
import traceback

# checked when the pipeline was built in section 1
print(pipeline.preflight_output_shapes)

# Create a new node probe with incorrect dimensions
probe = ModelNode("probe", nn.Linear(512, 2), inputs="student.logits", outputs="logits")

# Attempt to build and preflight the Execution Graph
try:
    ExecutionGraph([student, probe], preflight_sample=batch)  # 512 should be 10
except RuntimeError as error:
    # Output the Runtime Errors
    print("".join(traceback.format_exception_only(error)))

{'student.logits': (8, 10)}
RuntimeError: addmm: mat1 and mat2 shapes cannot be multiplied (8x10 and 512x2)
in node 'probe', reading student.logits (8, 10)



## 6. Record a run

Training code **emits** events; **sinks** decide where they go. 

`LocalFileSink` writes events within the run directory: 
- `manifest.json` stores library versions and the machine (written immediately);
- `status.json` (written immediately);
- `events.jsonl` stores each event as it happens; and
- `summary.json` (the last value of each scalar) at the end. 

`ConsoleSink` prints one line per event to the console standard output. 

`OutputRouter` sends every event to both and, used with `with`, records the final status: `completed`, `failed` (with the traceback) or `interrupted`. 

`RunEvent` is the class contianing event information. It can use several different datatypes, including Scalars. Scalars can be passed as 0-dim tensors, as the loss composer returns them. See docs for different 

In [ ]:
import json

from lwfm.results.events import RunEvent
from lwfm.results.router import OutputRouter
from lwfm.results.sinks import ConsoleSink, LocalFileSink

# Create a temporary directory for LocalFileSink to use
run_dir = Path(tempfile.mkdtemp()) / "demo_run"

# Create both a LocalFileSink and a ConsoleSink
output_sinks = [
    LocalFileSink(
        run_dir,
        # Manifest fields allow us to write extra lines in manifest.json
        # We can use name to include the source of the experiment
        manifest_fields={"name": "notebook_demo"},
    ),
    ConsoleSink(),
]

# Create an Output Router
with OutputRouter(*output_sinks) as router:
    # Train the pipeline over 3 steps
    pipeline.train()
    for step in range(3):
        # Compute the training loss, and take a step
        # Total is the total loss,
        #   and terms is a dictionary of individual loss components
        total, terms = loss_fn(batch, pipeline(batch))
        optimizer.zero_grad()
        total.backward()
        optimizer.step()

        # For each loss component, let's create an Output event
        for name, term in terms.items():
            # Create a Scalar Run Event (we're only storing a single loss value)
            router.write(
                RunEvent(
                    "scalar",  # Event "kind" stores the type
                    f"train/loss/{name}",  # Give the run event a name
                    term,  # The actual value to store
                    stage="fit",  # The current stage
                    step=step,
                )  # The current step
            )

[fit step 0] train/loss/ce = 0.0005448
[fit step 1] train/loss/ce = 0.0005282
[fit step 2] train/loss/ce = 0.0005128
status = completed


In [11]:
# Display the files in the temporary directory used by LocalFileSink
print(sorted(p.name for p in run_dir.iterdir()))

# Load status.json, and output the final status
print("status:", json.loads((run_dir / "status.json").read_text())["status"])

# Display the number of events stored
print("events:", len((run_dir / "events.jsonl").read_text().splitlines()))

# Display the last value of each scalar.
print("summary:", json.loads((run_dir / "summary.json").read_text()))

['events.jsonl', 'manifest.json', 'status.json', 'summary.json']
status: completed
events: 4
summary: {'train/loss/ce': 0.000512760947458446}


## 7. Evaluate top-1 and top-5

A metric is an unchanged torchmetrics object. 

A `MetricBinding` names the values its `update(preds, target)` reads, in order. 

`Evaluator().evaluate` resets every metric, runs the graph in eval mode under `torch.inference_mode()`, updates each metric per batch and computes once at the end, so accuracy is over all samples (micro), not an average of batch averages. The graph is put back in train mode afterwards, so the same call works for validation during training. 

Any plain list or `DataLoader` of batches works as the loader. With an `output_sink`, values are written as `eval/<split>/<name>`. Both scores are 1.0 here only because the model overfit this very batch in section 2.

In [ ]:
from torchmetrics.classification import MulticlassAccuracy

from lwfm.evaluation.evaluator import Evaluator
from lwfm.evaluation.metrics import MetricBinding

# Values MetricBinding will read
PRED_READS = ["student.logits", "targets.class"]  # update(preds, target)
metrics = [
    # Define a metric to use
    MetricBinding(
        f"top{k}",  # Name
        MulticlassAccuracy(num_classes=10, top_k=k, average="micro"),  # TorchMetric
        PRED_READS,
    )
    for k in (1, 5)  # Create each top1, top2, ... top5
]

# Supply validation batches here
# We can test on multiple batches, here we just use the same twice
#   just to test that we can use this for multiple in the future.
val_loader = [batch, batch]  # any iterable of batches; usually a DataLoader

result = Evaluator().evaluate(
    pipeline,
    val_loader,
    metrics,
    device=torch.device("cpu"),
    output_sink=ConsoleSink(),  # any OutputSink
)
print(result.metric_values, "| pipeline back in train mode:", pipeline.training)

eval/val/top1 = 1
eval/val/top5 = 1
{'top1': 1.0, 'top5': 1.0} | pipeline back in train mode: True
